# RAD gốc + SkinCAP — huấn luyện và đánh giá trên Google Colab

Notebook này là workflow Colab cho **RAD gốc** (`main_rad.py`) trên SkinCAP-47. Nó có thể tạo một lần chạy mới hoặc tiếp tục một lần chạy dang dở từ checkpoint đã lưu trên Google Drive.

1. Chuẩn bị đúng repository và dependency.
2. Kiểm tra GPU, Google Drive, ClinicalBERT và dữ liệu SkinCAP.
3. Xác thực một batch dữ liệu thật.
4. Tạo hoặc chọn một thư mục chạy, rồi ghi cấu hình 20 epoch.
5. Huấn luyện/tiếp tục `main_rad.py`.
6. Kiểm tra checkpoint, metrics và kết quả test của `best.pt`.

**Trước khi chạy:** chọn **Runtime → Change runtime type → GPU**. Không chạy lại tiền xử lý SkinCAP và không chạy các script Slurm trong `scripts/slurm/`.

> Mỗi thí nghiệm mới được đặt trong `RAD-outputs/skin-rad-20ep/YYYYMMDD_HHMMSS`. Để resume, giữ nguyên cả `EXPERIMENT_NAME` và `RUN_ID` của lần chạy đó.

## 1. Chuẩn bị Python và repository

Ô này clone repository nếu chưa có. Nếu đã có từ phiên Colab hiện tại, nó chỉ cập nhật metadata Git; không xóa output trên Drive.

In [ ]:
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version)

REPO_DIR = Path("/content/RAD-colab")
REPO_URL = "https://github.com/trisle0805/RAD-colab.git"
BRANCH = "RAD-colab-enhanced"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-3"], check=True)
assert (REPO_DIR / "main_rad.py").is_file(), "Sai repository: không thấy main_rad.py"

# Tương thích cả layout mới lẫn commit Colab cũ chưa chuyển file vào requirements/.
COLAB_REQUIREMENTS = next(
    (path for path in (
        REPO_DIR / "requirements/colab-skin.txt",
        REPO_DIR / "requirements-colab-skin.txt",
    ) if path.is_file()),
    None,
)
assert COLAB_REQUIREMENTS is not None, "Không tìm thấy file dependency Colab cho SkinCAP."
print(f"✅ Repository: {REPO_DIR} (nhánh {BRANCH})")
print("Requirements:", COLAB_REQUIREMENTS.relative_to(REPO_DIR))

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Cloning into '/content/RAD-colab'...
remote: Enumerating objects: 291, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 291 (delta 119), reused 266 (delta 94), pack-reused 0 (from 0)
Receiving objects: 100% (291/291), 1.39 MiB | 32.40 MiB/s, done.
Resolving deltas: 100% (119/119), done.


## 2. Cài dependency và kiểm tra GPU

`requirements/colab-skin.txt` không cài lại `torch`/`torchvision`, để giữ cặp CUDA tương thích mà Colab đang cung cấp.

In [ ]:
%cd /content/RAD-colab
%pip install -q -r {COLAB_REQUIREMENTS}

import importlib
from importlib.metadata import version

modules = {
    "yaml": "PyYAML",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "tqdm": "tqdm",
    "cv2": "opencv-python-headless",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "pydicom": "pydicom",
    "nibabel": "nibabel",
    "tensorboardX": "tensorboardX",
    "transformers": "transformers",
    "timm": "timm",
    "einops": "einops",
    "peft": "peft",
}

for module, package in modules.items():
    importlib.import_module(module)
    print(f"✅ {module:15} {version(package)}")

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

/content/RAD-colab
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 143.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
✅ yaml            6.0.3
✅ numpy           2.1.3
✅ pandas          2.2.3
✅ PIL             11.3.0
✅ tqdm 

## 3. Mount Drive, tải ClinicalBERT khi cần và chọn lần chạy

Chỉ sửa các đường dẫn riêng nếu vị trí dữ liệu hoặc output trên Drive thay đổi. Nếu `/content/models/Bio_ClinicalBERT` chưa tồn tại trong runtime Colab hiện tại, notebook sẽ tải model gốc `emilyalsentzer/Bio_ClinicalBERT` từ Hugging Face vào đó. Runtime Colab là tạm thời nên việc tải lại sau khi reset runtime là bình thường.

- Để `RUN_ID = None` khi chủ động bắt đầu một thí nghiệm mới. Notebook tạo và in một timestamp dạng `YYYYMMDD_HHMMSS`.
- Khi Colab ngắt hoặc reset runtime, dán timestamp đã in vào `RUN_ID` để tiếp tục **đúng** lần chạy đó.
- Không để `RUN_ID = None` khi muốn resume, vì điều này sẽ tạo thí nghiệm mới.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

SKINCAP_ROOT = Path("/content/drive/MyDrive/Cao học/Experiments/SkinCAP")
TRAIN_CSV = SKINCAP_ROOT / "skincap_47_train.csv"
VAL_CSV = SKINCAP_ROOT / "skincap_47_val.csv"
TEST_CSV = SKINCAP_ROOT / "skincap_47_test.csv"
IMAGE_ROOT = SKINCAP_ROOT / "skincap"

BERT_MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"
BERT_DIR = Path("/content/models/Bio_ClinicalBERT")
if not (BERT_DIR / "config.json").is_file():
    from huggingface_hub import snapshot_download

    BERT_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"Đang tải {BERT_MODEL_ID} vào {BERT_DIR}...")
    snapshot_download(repo_id=BERT_MODEL_ID, local_dir=str(BERT_DIR))
else:
    print(f"✅ Đã có ClinicalBERT: {BERT_DIR}")

# None: tạo một lần chạy mới. Muốn resume hoặc chỉ xem kết quả cũ: dán RUN_ID vào đây.
EXPERIMENT_NAME = "skin-rad-20ep"
RUNS_ROOT = Path("/content/drive/MyDrive/Cao học/RAD-outputs") / EXPERIMENT_NAME
RUN_ID = None

for label, path, check in [
    ("Train CSV", TRAIN_CSV, Path.is_file),
    ("Validation CSV", VAL_CSV, Path.is_file),
    ("Test CSV", TEST_CSV, Path.is_file),
    ("Thư mục ảnh", IMAGE_ROOT, Path.is_dir),
    ("ClinicalBERT", BERT_DIR, Path.is_dir),
]:
    assert check(path), f"Không tìm thấy {label}: {path}"
    print(f"✅ {label}: {path}")

png_count = len(list(IMAGE_ROOT.glob("*.png")))
print("Số ảnh PNG:", png_count)
assert png_count > 0, "Không có ảnh PNG trong IMAGE_ROOT."

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    RUN_DIR = RUNS_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=False, exist_ok=False)
    print(f"✅ Tạo lần chạy mới: {RUN_ID} (UTC)")
    print("⚠️ Lưu EXPERIMENT_NAME và RUN_ID này để resume sau khi runtime reset.")
else:
    RUN_DIR = RUNS_ROOT / RUN_ID
    assert RUN_DIR.is_dir(), f"Không tìm thấy run trên Drive: {RUN_DIR}"
    latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
    best_checkpoint = RUN_DIR / "checkpoints" / "best.pt"
    final_metrics = RUN_DIR / "final_metrics.json"
    print(f"✅ Dùng run có sẵn: {RUN_ID}")
    print("Latest checkpoint:", latest_checkpoint if latest_checkpoint.is_file() else "chưa có")
    print("Best checkpoint:", best_checkpoint if best_checkpoint.is_file() else "chưa có")
    print("Final test metrics:", final_metrics if final_metrics.is_file() else "chưa có")

for directory in ("checkpoints", "metrics", "predictions", "tensorboard", "logs"):
    (RUN_DIR / directory).mkdir(exist_ok=True)

print("Run directory:", RUN_DIR)

Mounted at /content/drive
Đang tải emilyalsentzer/Bio_ClinicalBERT vào /content/models/Bio_ClinicalBERT...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

graph.pbtxt: 0.00B [00:00, ?B/s]

...

flax_model.msgpack:   0%|          | 0.00/433M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/527M [00:00<?, ?B/s]

✅ Train CSV: /content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap_47_train.csv
✅ Validation CSV: /content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap_47_val.csv
✅ Test CSV: /content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap_47_test.csv
✅ Thư mục ảnh: /content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap
✅ ClinicalBERT: /content/models/Bio_ClinicalBERT
Số ảnh PNG: 4000
✅ Resume lần chạy: 20261005_091021
Run directory: /content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021


## 4. Kiểm tra ClinicalBERT và một batch SkinCAP

Ô này không train. Nó xác nhận model BERT cục bộ có hidden size 768 và loader đọc đúng CSV, ảnh và **47 nhãn** trước khi bắt đầu thí nghiệm dài.

In [ ]:
%cd /content/RAD-colab

from transformers import AutoConfig, AutoTokenizer
from torch.utils.data import DataLoader
from dataset.dataset_entity import Skin_Train_Dataset
from dataset.test_dataset import Skin_Test_Dataset

bert_config = AutoConfig.from_pretrained(BERT_DIR, local_files_only=True)
AutoTokenizer.from_pretrained(BERT_DIR, local_files_only=True)
assert bert_config.hidden_size == 768, (
    f"ClinicalBERT phải có hidden_size=768, nhận được {bert_config.hidden_size}."
)
print("✅ ClinicalBERT:", bert_config.model_type, "hidden_size=", bert_config.hidden_size)

train_dataset = Skin_Train_Dataset(TRAIN_CSV, image_res=512, image_root=IMAGE_ROOT)
val_dataset = Skin_Test_Dataset(VAL_CSV, image_res=512, image_root=IMAGE_ROOT)
test_dataset = Skin_Test_Dataset(TEST_CSV, image_res=512, image_root=IMAGE_ROOT)
batch = next(iter(DataLoader(train_dataset, batch_size=2, num_workers=0)))

print("Số mẫu train:", len(train_dataset))
print("Số mẫu val :", len(val_dataset))
print("Số mẫu test :", len(test_dataset))
print("Batch image :", tuple(batch["image"].shape))
print("Batch label :", tuple(batch["label"].shape))
assert tuple(batch["image"].shape) == (2, 3, 512, 512)
assert tuple(batch["label"].shape) == (2, 47)
print("✅ Dữ liệu SkinCAP sẵn sàng.")

/content/RAD-colab
✅ ClinicalBERT: bert hidden_size= 768
Số mẫu train: 1683
Số mẫu val : 297
Số mẫu test : 546
Batch image : (2, 3, 512, 512)
Batch label : (2, 47)
✅ Dữ liệu SkinCAP sẵn sàng.


## 5. Tạo cấu hình huấn luyện RAD gốc

Cấu hình dưới đây là một thí nghiệm SkinCAP-47 mới, rõ ràng và có thể tái lập: `batch_size=4`, `grad_accumulation_steps=4`, **20 epoch**, `warmup_epochs=10`. Giá trị này được ghi vào chính `RUN_DIR`; khi resume notebook luôn dùng lại file đó, không tạo cấu hình mới khác với checkpoint đang tiếp tục.

In [ ]:
import yaml

FULL_CONFIG = RUN_DIR / "rad-skin-20ep.yaml"
if FULL_CONFIG.is_file():
    config = yaml.safe_load(FULL_CONFIG.read_text(encoding="utf-8"))
    print(f"✅ Dùng lại cấu hình đã lưu của run: {FULL_CONFIG}")
else:
    with (REPO_DIR / "configs/skin.yaml").open(encoding="utf-8") as file:
        config = yaml.safe_load(file)

    config.update({
        "batch_size": 4,
        "test_batch_size": 4,
        "num_workers": 2,
        "test_num_workers": 2,
        "grad_accumulation_steps": 4,
    })
    config["schedular"].update({
        "epochs": 20,
        "warmup_epochs": 10,
    })
    FULL_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    print(f"✅ Đã tạo cấu hình: {FULL_CONFIG}")

assert config["image_res"] == 512
assert config["batch_size"] > 0 and config["test_batch_size"] > 0
assert config["schedular"]["epochs"] > 0
print(FULL_CONFIG.read_text(encoding="utf-8"))

ICD_train_file: ''
ICD_test_file: ''
image_root: ''
image_res: 512
batch_size: 4
test_batch_size: 4
grad_accumulation_steps: 4
num_workers: 2
test_num_workers: 2
temp: 0.07
mlm_probability: 0.15
queue_size: 8192
momentum: 0.995
alpha: 0.4
optimizer:
  opt: adamw
  lr: 0.0001
  weight_decay: 0.01
schedular:
  sched: cosine
  lr: 5.0e-05
  epochs: 20
  min_lr: 1.0e-06
  decay_rate: 1
  warmup_lr: 1.0e-06
  warmup_epochs: 10
  cooldown_epochs: 0



## 6. Khởi chạy hoặc resume an toàn

Chạy ô này để bắt đầu hoặc tiếp tục lần chạy đã chọn ở bước 3.

Sau mỗi **epoch hoàn chỉnh** (train + validation + metrics), source sẽ lưu trên Drive:

- `checkpoints/latest.pt`: checkpoint mới nhất, ghi đè an toàn để resume từ epoch kế tiếp.
- `checkpoints/epoch_010.pt`, `epoch_020.pt`, `epoch_030.pt`: checkpoint mốc.
- `metrics/metrics_history.csv` và `metrics/latest_metrics.json`: lịch sử và metrics mới nhất.
- `predictions/epoch_XXX.npz`: dữ liệu prediction nén cho mỗi epoch.
- `logs/training.log` và `tensorboard/`: log bền vững.

Nếu Colab ngắt giữa một epoch, chỉ trạng thái của epoch hoàn chỉnh gần nhất được dùng để resume. Không tạo `RUN_ID` mới khi resume.

In [ ]:
import os
import subprocess
import sys

latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
if latest_checkpoint.is_file():
    print("✅ Sẽ resume từ checkpoint hoàn chỉnh:", latest_checkpoint)
else:
    print("ℹ️ Chưa có checkpoint: thí nghiệm sẽ bắt đầu từ epoch 1.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.chdir(REPO_DIR)

# subprocess nhận từng tham số riêng, nên các đường dẫn có khoảng trắng hoặc dấu tiếng Việt luôn an toàn.
command = [
    sys.executable,
    "main_rad.py",
    "--config", str(FULL_CONFIG),
    "--dataset", "skin",
    "--train_csv", str(TRAIN_CSV),
    "--val_csv", str(VAL_CSV),
    "--test_csv", str(TEST_CSV),
    "--image_root", str(IMAGE_ROOT),
    "--output_dir", str(RUN_DIR),
    "--bert_model_name", str(BERT_DIR),
    "--guideline_path", "./guideline/qwen_maxtoken2k_skincap47_4sources.jsonl",
    "--max_length", "512",
    "--topk", "1", "2", "3",
    "--embed_dim", "768",
    "--loss_ratio", "0.1",
    "--contrast_ratio_text", "0.1",
    "--contrast_ratio_vision", "0.001",
]

print("Lệnh sẽ chạy:")
print(" ".join(repr(part) if " " in part else part for part in command))
subprocess.run(command, check=True)

✅ Sẽ resume từ checkpoint hoàn chỉnh: /content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021/checkpoints/latest.pt
Lệnh sẽ chạy:
/usr/bin/python3 main_rad.py --config '/content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021/skin-full-30ep.yaml' --dataset skin --train_csv '/content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap_47_train.csv' --val_csv '/content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap_47_val.csv' --test_csv '/content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap_47_test.csv' --image_root '/content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap' --output_dir '/content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021' --bert_model_name /content/models/Bio_ClinicalBERT --guideline_path ./guideline/qwen_maxtoken2k_skincap47_4sources.jsonl --max_length 512 --topk 1 2 3 --embed_dim 768 --loss_ratio 0.1 --contrast_ratio_text 0.1 --contrast_ratio_vision 0.001


CompletedProcess(args=['/usr/bin/python3', 'main_rad.py', '--config', '/content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021/skin-full-30ep.yaml', '--dataset', 'skin', '--train_csv', '/content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap_47_train.csv', '--val_csv', '/content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap_47_val.csv', '--test_csv', '/content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap_47_test.csv', '--image_root', '/content/drive/MyDrive/Cao học/Experiments/SkinCAP/skincap', '--output_dir', '/content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021', '--bert_model_name', '/content/models/Bio_ClinicalBERT', '--guideline_path', './guideline/qwen_maxtoken2k_skincap47_4sources.jsonl', '--max_length', '512', '--topk', '1', '2', '3', '--embed_dim', '768', '--loss_ratio', '0.1', '--contrast_ratio_text', '0.1', '--contrast_ratio_vision', '0.001'], returncode=0)

## 7. Kiểm tra trạng thái đã lưu trên Drive

Sau khi một epoch hoàn thành, chạy ô này để xem checkpoint có thể resume, metrics mới nhất và phần cuối của log. Sau khi train hoàn tất, `main_rad.py` sẽ nạp `checkpoints/best.pt` và ghi `final_metrics.json` cùng `predictions/test_best.npz`. Ô này chỉ đọc file; không bắt đầu huấn luyện.

In [ ]:
import json
import pandas as pd

latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
history_path = RUN_DIR / "metrics" / "metrics_history.csv"
latest_metrics_path = RUN_DIR / "metrics" / "latest_metrics.json"
log_path = RUN_DIR / "logs" / "training.log"
best_checkpoint = RUN_DIR / "checkpoints" / "best.pt"
final_metrics_path = RUN_DIR / "final_metrics.json"
predictions_path = RUN_DIR / "predictions" / "test_best.npz"

print("Run directory:", RUN_DIR)
print("Latest checkpoint:", latest_checkpoint if latest_checkpoint.is_file() else "chưa có")
print("Best checkpoint:", best_checkpoint if best_checkpoint.is_file() else "chưa có")
print("Final test predictions:", predictions_path if predictions_path.is_file() else "chưa có")
print("Milestone checkpoints:", [path.name for path in sorted((RUN_DIR / "checkpoints").glob("epoch_*.pt"))])

if history_path.is_file():
    history = pd.read_csv(history_path)
    display(history.tail(5))
else:
    print("ℹ️ Chưa có metrics_history.csv; epoch đầu tiên chưa hoàn thành.")

if latest_metrics_path.is_file():
    print("\nLatest metrics:")
    print(json.dumps(json.loads(latest_metrics_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

if log_path.is_file():
    print("\n20 dòng log cuối:")
    print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))
if final_metrics_path.is_file():
    print("\nFinal test metrics:")
    print(json.dumps(json.loads(final_metrics_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
else:
    print("\nChưa có final_metrics.json; file này chỉ được tạo sau khi train xong và test best.pt một lần.")

Run directory: /content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021
Latest checkpoint: /content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021/checkpoints/latest.pt
Best checkpoint: /content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021/checkpoints/best.pt
Milestone checkpoints: ['epoch_010.pt', 'epoch_020.pt']


,epoch,completed_at_utc,train_loss,train_loss_ce,train_loss_clip,learning_rate,label_mean_auc,label_mean_ap,label_mean_accuracy,label_mean_f1,...,guideline_macro_mrr_at_2,guideline_top3_accuracy,guideline_mrr_at_3,guideline_macro_mrr_at_3,guideline_validation_score,best_validation_score,selected_stream,prediction_file,validation_score,best_epoch
15,16,2026-10-05T15:44:20.792156+00:00,0.133935,0.020776,0.693594,0.000086,0.957693,0.803110,0.993839,0.840089,...,0.772892,0.845118,0.804153,0.781529,0.743426,0.790066,label,predictions/val_epoch_016.npz,0.781920,14
16,17,2026-10-05T15:59:51.954347+00:00,0.125758,0.020249,0.616721,0.000080,0.971528,0.820224,0.994341,0.845504,...,0.771582,0.841751,0.799663,0.779429,0.751684,0.790066,label,predictions/val_epoch_017.npz,0.774537,14
17,18,2026-10-05T16:15:23.988266+00:00,0.118554,0.015775,0.590249,0.000073,0.963813,0.805096,0.993982,0.844419,...,0.779388,0.835017,0.806958,0.785022,0.758485,0.790066,guideline,predictions/val_epoch_018.npz,0.758485,14
18,19,2026-10-05T16:30:58.572375+00:00,0.107723,0.011996,0.520323,0.000066,0.964206,0.790161,0.994054,0.825685,...,0.783770,0.838384,0.809764,0.787935,0.757870,0.790066,label,predictions/val_epoch_019.npz,0.779115,14
19,20,2026-10-05T16:46:33.234537+00:00,0.103390,0.009130,0.507515,0.000058,0.957562,0.791359,0.994412,0.832728,...,0.800926,0.858586,0.827160,0.808669,0.776632,0.790066,label,predictions/val_epoch_020.npz,0.780815,14



Latest metrics:
{
  "epoch": 20,
  "completed_at_utc": "2026-10-05T16:46:33.234537+00:00",
  "train_loss": 0.10339,
  "train_loss_ce": 0.00913,
  "train_loss_clip": 0.507515,
  "learning_rate": 5.8243506019491436e-05,
  "label_mean_auc": 0.9575617837748123,
  "label_mean_ap": 0.7913592725740072,
  "label_mean_accuracy": 0.9944122071781645,
  "label_mean_f1": 0.8327275973235984,
  "label_mean_precision": 0.9254150105213935,
  "label_mean_recall": 0.7847579134813178,
  "label_subset_accuracy": 0.797979797979798,
  "label_macro_f1": 0.7808154698803793,
  "label_balanced_accuracy": 0.7804772650517332,
  "label_top1_accuracy": 0.8013468013468014,
  "label_top2_accuracy": 0.8316498316498316,
  "label_mrr_at_2": 0.8164983164983165,
  "label_macro_mrr_at_2": 0.7993138067606153,
  "label_top3_accuracy": 0.8484848484848485,
  "label_mrr_at_3": 0.8221099887766555,
  "label_macro_mrr_at_3": 0.8042783457677073,
  "label_validation_score": 0.7808154698803793,
  "guideline_mean_auc": 0.9631159564679

## 8. Xuất attention RAD từ `best.pt`

Chạy phần này **sau khi** mục 7 xác nhận có `best.pt`, `final_metrics.json` và `predictions/test_best.npz`. Exporter chỉ đọc checkpoint tốt nhất, replay đúng test set rồi đối chiếu xác suất với prediction đã lưu trước khi ghi evidence. Nó không huấn luyện lại hay thay đổi model.

Kết quả được lưu tại `RUN_DIR/checklist/rad_attention_evidence.npz`. Để tạo bản đồ attention đầy đủ cho một số ca cố định, đặt `FULL_ATTENTION_SAMPLES` là các chỉ số test cần lưu. Nếu để `[]`, HTML ở mục 9 vẫn chưa thể tạo heatmap đầy đủ.

In [ ]:
import subprocess
import sys

CHECKLIST_DIR = RUN_DIR / "checklist"
RAD_EVIDENCE_PATH = CHECKLIST_DIR / "rad_attention_evidence.npz"
RAD_SCHEMA_PATH = CHECKLIST_DIR / "rad_attention_evidence.schema.json"

assert best_checkpoint.is_file(), f"Không tìm thấy best.pt: {best_checkpoint}"
assert final_metrics_path.is_file(), f"Không tìm thấy final_metrics.json: {final_metrics_path}"
assert predictions_path.is_file(), f"Không tìm thấy test_best.npz: {predictions_path}"

# Chọn các chỉ số test cần lưu full attention để tạo heatmap HTML ở mục 9.
# Có thể đổi thành các chỉ số quan tâm, ví dụ [0, 30, 70].
FULL_ATTENTION_SAMPLES = [0, 1, 2]
assert all(index >= 0 for index in FULL_ATTENTION_SAMPLES), "Chỉ số mẫu phải không âm."

command = [
    sys.executable,
    "proposition/visualization/export_rad_attention.py",
    "--rad_run_dir", str(RUN_DIR),
    "--rad_config", str(FULL_CONFIG),
    "--test_csv", str(TEST_CSV),
    "--image_root", str(IMAGE_ROOT),
    "--bert_model_name", str(BERT_DIR),
    "--guideline_path", "guideline/qwen_maxtoken2k_skincap47_4sources.jsonl",
    "--output_dir", str(CHECKLIST_DIR),
    "--batch_size", str(config["test_batch_size"]),
    "--attention_topk", "10",
]

if FULL_ATTENTION_SAMPLES:
    command.extend(["--full_attention_indices", *(str(index) for index in FULL_ATTENTION_SAMPLES)])

print("Export attention RAD từ best.pt:")
print(" ".join(repr(part) if " " in part else part for part in command))
subprocess.run(command, cwd=REPO_DIR, check=True)

assert RAD_EVIDENCE_PATH.is_file(), f"Không tạo được evidence: {RAD_EVIDENCE_PATH}"
assert RAD_SCHEMA_PATH.is_file(), f"Không tạo được schema: {RAD_SCHEMA_PATH}"
print("✅ Evidence RAD:", RAD_EVIDENCE_PATH)
print(RAD_SCHEMA_PATH.read_text(encoding="utf-8"))

## 9. Tạo báo cáo HTML định tính cho RAD gốc

Phần này chạy renderer chỉ đọc evidence ở mục 8, tạo một file HTML cho mỗi ca có full attention và nhúng report đầu tiên vào notebook. Báo cáo bao gồm ảnh, caption, top-3 chẩn đoán fused, xác suất hai nhánh RAD, heatmap attention ảnh và attention caption của nhánh `guideline`/`label`.

RAD gốc không có “mệnh đề” như mô hình proposition; bằng chứng ở đây là hai attention map của decoder. `PAD mass` được hiển thị vì RAD gốc không truyền padding mask vào decoder.

In [ ]:
import numpy as np
from IPython.display import IFrame, display

REPORT_ROOT = CHECKLIST_DIR / "rad_attention_reports"
with np.load(RAD_EVIDENCE_PATH, allow_pickle=False) as evidence:
    available_samples = evidence["full_attention_sample_indices"].astype(int).tolist()
assert available_samples, "Không có full attention. Hãy chọn ít nhất một index ở mục 8 rồi export lại."

command = [
    sys.executable,
    "proposition/visualization/render_rad_attention.py",
    "--evidence", str(RAD_EVIDENCE_PATH),
    "--bert_model_name", str(BERT_DIR),
    "--output_dir", str(REPORT_ROOT),
    "--samples", *(str(index) for index in available_samples),
]
subprocess.run(command, cwd=REPO_DIR, check=True)

report_paths = [REPORT_ROOT / f"sample_{index:04d}" / "report.html" for index in available_samples]
for path in report_paths:
    assert path.is_file(), f"Không tạo được report: {path}"
    print("✅ HTML:", path)

display(IFrame(src=str(report_paths[0]), width="100%", height=1600))